# EXP029: DQN-Param-Raw（MLE・単一非相関バンク）

EXP023 の共有 DQN-Param-Raw 構造を維持し、能力推定をEXP029と同じ
MLE実装へ置き換える。

- state: `[theta_hat_MLE, t/L]`
- item features: 標準化した `[a, b, c]`
- reward: 選択前の推定値における Fisher 情報量 `I_i(theta_hat_MLE)`
- MLE: `[-4, 4]` の有界最適化。全正答・全誤答時はEXP029の段階的更新
- TD target: 残存項目マスクと終端フラグを使用
- default bank: `data/uncorrelated_banks/item_bank_uncor_1.csv`



In [1]:
from __future__ import annotations

import copy
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, cast

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.optimize import minimize_scalar


def find_project_root() -> Path:
    """Find the repository root in local and Colab environments."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    candidates.extend(
        [
            Path("/content/Grad_Research"),
            Path("/content/drive/MyDrive/Grad_Research"),
            Path("/content/drive/MyDrive/Colab Notebooks/Grad_Research"),
        ]
    )

    for root in candidates:
        if (root / "data").is_dir() and (root / "EXP029").is_dir():
            return root

    raise FileNotFoundError(
        "Could not find the project root. Run this notebook inside the repository."
    )


def choose_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


ROOT = find_project_root()
EXP029_DIR = ROOT / "EXP029"
MODEL_DIR = EXP029_DIR / "models"
RESULTS_DIR = EXP029_DIR / "results"
DEVICE = choose_device()

print(f"Device      : {DEVICE}")
print(f"Project root: {ROOT}")
print(f"Model dir   : {MODEL_DIR}")
print(f"Results dir : {RESULTS_DIR}")

Device      : mps
Project root: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History
Model dir   : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP029/models
Results dir : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP029/results


## Configuration

`seed` は NumPy、Python、PyTorch に共通して適用する。
MLEの初期値はEXP029と同じく `Uniform(-0.5, 0.5)` から生成する。



In [ ]:
@dataclass
class Config:
    # Network: [theta_hat, t/L, a, b, c] -> 64 -> 64 -> 1
    state_size: int = 2
    item_feature_size: int = 3
    first_hidden: int = 64
    second_hidden: int = 64

    # Training (inherited from EXP023 unless noted otherwise)
    test_length: int = 40
    gamma: float = 0.1
    memory_capacity: int = 1000
    epsilon: float = 0.1
    batch_size: int = 128
    q_network_iteration: int = 40
    learning_rate: float = 1e-3
    training_size: int = 1000
    validation_size: int = 200
    validation_interval: int = 50
    seed: int = 42

    # Bank / examinee distribution
    bank_type: str = "uncor"
    bank_id: int = 1
    prior: str = "normal"
    n_items: int = 500

    @property
    def input_size(self) -> int:
        return self.state_size + self.item_feature_size

    def validate(self) -> None:
        if self.state_size != 2:
            raise ValueError("EXP029 requires state_size=2: [theta_hat, t/L].")
        if self.item_feature_size != 3:
            raise ValueError("EXP029 Raw requires item_feature_size=3: [a, b, c].")
        if not 0.0 <= self.epsilon <= 1.0:
            raise ValueError("epsilon must be between 0 and 1.")
        if self.test_length > self.n_items:
            raise ValueError("test_length cannot exceed n_items without replacement.")
        if self.batch_size > self.memory_capacity:
            raise ValueError("batch_size cannot exceed memory_capacity.")
        if self.validation_interval > self.training_size:
            raise ValueError(
                "validation_interval must not exceed training_size; otherwise no "
                "checkpoint is selected."
            )


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


cfg = Config()
cfg.validate()
set_seed(cfg.seed)
print(cfg)

Config(state_size=2, item_feature_size=3, first_hidden=64, second_hidden=64, test_length=40, gamma=0.1, memory_capacity=1000, epsilon=0.1, batch_size=128, q_network_iteration=40, learning_rate=0.001, training_size=1000, validation_size=200, validation_interval=50, seed=42, bank_type='uncor', bank_id=1, prior='normal', n_items=500)


## 3PL response, Fisher information, and MLE estimation



In [3]:
def respond(
    item_para: np.ndarray, theta: np.ndarray | float, d: float = 1.0
) -> np.ndarray:
    """Generate independent Bernoulli responses for the supplied items."""
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    probability = c + (1.0 - c) / (1.0 + np.exp(-d * a * (theta - b)))
    return (np.random.random(size=probability.shape) <= probability).astype(np.int64)


def fisher_information(
    item_para: np.ndarray, theta: np.ndarray | float, d: float = 1.0
) -> np.ndarray:
    """Calculate 3PL Fisher information at theta."""
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    return (
        d**2
        * a**2
        * (1.0 - c)
        / (c + np.exp(d * a * (theta - b)))
        / (1.0 + np.exp(-d * a * (theta - b))) ** 2
    )


def mle(item_paras: np.ndarray, responses: np.ndarray, d: float = 1.0) -> float:
    """Estimate theta by the bounded MLE used in EXP029."""
    a = item_paras[:, 0]
    b = item_paras[:, 1]
    c = item_paras[:, 2]

    def negative_log_likelihood(theta: float) -> float:
        probability = (1.0 - c) / (1.0 + np.exp(-d * a * (theta - b))) + c
        probability = np.clip(probability, 1e-10, 1.0 - 1e-10)
        return float(
            -np.sum(
                responses * np.log(probability)
                + (1 - responses) * np.log(1.0 - probability)
            )
        )

    result = cast(
        Any,
        minimize_scalar(negative_log_likelihood, bounds=(-4.0, 4.0), method="bounded"),
    )
    return float(result.x)


def estimate_theta_single(
    raw_item_bank: np.ndarray,
    administered_items: np.ndarray,
    responses: np.ndarray,
    current_theta: float,
) -> float:
    if np.all(responses == 1):
        return current_theta + (raw_item_bank[:, 1].max() - current_theta) / 2.0
    if np.all(responses == 0):
        return current_theta - (current_theta - raw_item_bank[:, 1].min()) / 2.0
    return mle(administered_items, responses)

## Item-feature standardization

単一学習バンク全体から `a`, `b`, `c` の平均と母標準偏差（`ddof=0`）を
計算する。Fisher情報量と反応生成には元の項目パラメータを使用し、Qネットワーク
の入力だけを標準化する。



In [4]:
def standardize_item_features(
    item_bank: np.ndarray,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    item_mean = item_bank.mean(axis=0)
    item_std = item_bank.std(axis=0, ddof=0)
    if np.any(item_std <= 0.0):
        raise ValueError("All item-feature standard deviations must be positive.")
    standardized = (item_bank - item_mean) / item_std
    return standardized.astype(np.float32), item_mean, item_std


def load_experiment_data(config: Config) -> tuple[np.ndarray, np.ndarray]:
    bank_dir = {
        "uncor": ROOT / "data" / "uncorrelated_banks",
        "cor": ROOT / "data" / "correlated_banks",
    }.get(config.bank_type)
    if bank_dir is None:
        raise ValueError(f"Unsupported bank_type: {config.bank_type!r}.")

    bank_path = bank_dir / f"item_bank_{config.bank_type}_{config.bank_id}.csv"
    theta_path = ROOT / "data" / "theta_true" / f"theta_true_{config.bank_id}.csv"
    bank_frame = pd.read_csv(bank_path)
    item_bank = bank_frame.loc[:, ["a", "b", "c"]].to_numpy(dtype=np.float64)
    item_bank = item_bank[: config.n_items]
    theta_test = pd.read_csv(theta_path).loc[:, "x"].to_numpy(dtype=np.float64)

    if item_bank.shape != (config.n_items, config.item_feature_size):
        raise ValueError(
            f"Expected item bank shape {(config.n_items, config.item_feature_size)}, "
            f"got {item_bank.shape}."
        )
    return item_bank, theta_test


item_bank, theta_test = load_experiment_data(cfg)
item_features, item_feature_mean, item_feature_std = standardize_item_features(
    item_bank
)
action_space = item_bank.shape[0]
item_features_tensor = torch.from_numpy(item_features).to(DEVICE)

print(f"item bank       : {item_bank.shape}")
print(f"theta test      : {theta_test.shape}")
print(f"item mean [abc] : {item_feature_mean}")
print(f"item std  [abc] : {item_feature_std}")

item bank       : (500, 3)
theta test      : (5000,)
item mean [abc] : [ 1.18845488e+00 -3.01400005e-04  2.47336919e-01]
item std  [abc] : [0.24811954 1.01221578 0.0208518 ]


## Shared DQN-Param-Raw Q-network

同じMLPを全候補項目に共有し、出力形状を `[batch, n_items]` とする。



In [5]:
class DQNParamRaw(nn.Module):
    def __init__(
        self,
        state_size: int,
        item_feature_size: int,
        first_hidden: int,
        second_hidden: int,
    ) -> None:
        super().__init__()
        self.state_size = state_size
        self.item_feature_size = item_feature_size
        self.q_network = nn.Sequential(
            nn.Linear(state_size + item_feature_size, first_hidden),
            nn.ReLU(),
            nn.Linear(first_hidden, second_hidden),
            nn.ReLU(),
            nn.Linear(second_hidden, 1),
        )

    def forward(
        self, states: torch.Tensor, standardized_items: torch.Tensor
    ) -> torch.Tensor:
        if states.ndim != 2 or states.shape[1] != self.state_size:
            raise ValueError(
                f"states must have shape [B, {self.state_size}], got {states.shape}."
            )
        if (
            standardized_items.ndim != 2
            or standardized_items.shape[1] != self.item_feature_size
        ):
            raise ValueError(
                "standardized_items must have shape "
                f"[J, {self.item_feature_size}], got {standardized_items.shape}."
            )

        batch_size = states.shape[0]
        n_items = standardized_items.shape[0]
        expanded_states = states.unsqueeze(1).expand(-1, n_items, -1)
        expanded_items = standardized_items.unsqueeze(0).expand(batch_size, -1, -1)
        q_input = torch.cat((expanded_states, expanded_items), dim=-1)
        return self.q_network(q_input).squeeze(-1)

    def initialize(self) -> None:
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
                if module.bias is not None:
                    nn.init.zeros_(module.bias)


class ReplayBuffer:
    def __init__(self, capacity: int, state_size: int, n_items: int) -> None:
        self.capacity = capacity
        self.states = np.zeros((capacity, state_size), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.next_states = np.zeros((capacity, state_size), dtype=np.float32)
        self.terminals = np.zeros(capacity, dtype=np.bool_)
        self.next_available = np.zeros((capacity, n_items), dtype=np.bool_)
        self.counter = 0

    def __len__(self) -> int:
        return min(self.counter, self.capacity)

    def add(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        terminal: bool,
        next_available: np.ndarray,
    ) -> None:
        index = self.counter % self.capacity
        self.states[index] = state
        self.actions[index] = action
        self.rewards[index] = reward
        self.next_states[index] = next_state
        self.terminals[index] = terminal
        self.next_available[index] = next_available
        self.counter += 1

    def sample(
        self, batch_size: int
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        if len(self) < batch_size:
            raise ValueError("Not enough transitions to sample a full batch.")
        indices = np.random.choice(len(self), batch_size)
        return (
            self.states[indices],
            self.actions[indices],
            self.rewards[indices],
            self.next_states[indices],
            self.terminals[indices],
            self.next_available[indices],
        )

## Action selection and masked TD update



In [6]:
def choose_action(
    model: DQNParamRaw,
    state: np.ndarray,
    available: np.ndarray,
    standardized_items: torch.Tensor,
    epsilon: float,
) -> int:
    available_ids = np.flatnonzero(available)
    if available_ids.size == 0:
        raise ValueError("No item is available for selection.")
    if np.random.rand() < epsilon:
        return int(np.random.choice(available_ids))

    with torch.no_grad():
        state_tensor = (
            torch.from_numpy(state.astype(np.float32)).unsqueeze(0).to(DEVICE)
        )
        q_values = model(state_tensor, standardized_items).squeeze(0)
        available_tensor = torch.from_numpy(available).to(DEVICE)
        q_values = q_values.masked_fill(~available_tensor, -torch.inf)
        return int(q_values.argmax().item())


def choose_actions_greedy(
    model: DQNParamRaw,
    states: np.ndarray,
    available: np.ndarray,
    standardized_items: torch.Tensor,
) -> np.ndarray:
    if np.any(~available.any(axis=1)):
        raise ValueError("At least one examinee has no available item.")
    with torch.no_grad():
        state_tensor = torch.from_numpy(states.astype(np.float32)).to(DEVICE)
        available_tensor = torch.from_numpy(available).to(DEVICE)
        q_values = model(state_tensor, standardized_items)
        q_values = q_values.masked_fill(~available_tensor, -torch.inf)
        return q_values.argmax(dim=1).cpu().numpy()


def optimize_dqn(
    config: Config,
    eval_net: DQNParamRaw,
    target_net: DQNParamRaw,
    optimizer: optim.Optimizer,
    loss_function: nn.Module,
    replay_buffer: ReplayBuffer,
    standardized_items: torch.Tensor,
) -> float:
    (
        states,
        actions,
        rewards,
        next_states,
        terminals,
        next_available,
    ) = replay_buffer.sample(config.batch_size)

    states_tensor = torch.from_numpy(states).to(DEVICE)
    actions_tensor = torch.from_numpy(actions).to(DEVICE)
    rewards_tensor = torch.from_numpy(rewards).to(DEVICE)
    next_states_tensor = torch.from_numpy(next_states).to(DEVICE)
    terminals_tensor = torch.from_numpy(terminals).to(DEVICE)
    next_available_tensor = torch.from_numpy(next_available).to(DEVICE)

    q_eval = (
        eval_net(states_tensor, standardized_items)
        .gather(1, actions_tensor.unsqueeze(1))
        .squeeze(1)
    )

    with torch.no_grad():
        q_next = target_net(next_states_tensor, standardized_items)
        q_next_max = torch.zeros(config.batch_size, device=DEVICE)
        nonterminal = ~terminals_tensor
        if nonterminal.any():
            available_nonterminal = next_available_tensor[nonterminal]
            if torch.any(~available_nonterminal.any(dim=1)):
                raise RuntimeError(
                    "A nonterminal replay transition has no available item."
                )
            masked_q_next = q_next[nonterminal].masked_fill(
                ~available_nonterminal, -torch.inf
            )
            q_next_max[nonterminal] = masked_q_next.max(dim=1).values

        q_target = rewards_tensor + (
            config.gamma * (~terminals_tensor).float() * q_next_max
        )

    loss = loss_function(q_eval, q_target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return float(loss.item())

## Greedy validation and test simulation



In [7]:
def simulate_greedy_cat(
    config: Config,
    model: DQNParamRaw,
    raw_item_bank: np.ndarray,
    standardized_items: torch.Tensor,
    true_theta: np.ndarray,
    response_seed: int | None = None,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    if response_seed is not None:
        np.random.seed(response_seed)
    n_examinees = len(true_theta)
    n_items = raw_item_bank.shape[0]
    states = np.column_stack(
        (
            np.random.rand(n_examinees).astype(np.float32) - 0.5,
            np.zeros(n_examinees, dtype=np.float32),
        )
    )
    available = np.ones((n_examinees, n_items), dtype=np.bool_)
    selected_items = np.zeros((config.test_length, n_examinees), dtype=np.int64)
    responses = np.zeros((config.test_length, n_examinees), dtype=np.int64)
    theta_estimates = np.zeros((config.test_length, n_examinees), dtype=np.float64)

    model.eval()
    for step in range(config.test_length):
        actions = choose_actions_greedy(model, states, available, standardized_items)
        selected_items[step] = actions
        responses[step] = respond(raw_item_bank[actions], true_theta)
        available[np.arange(n_examinees), actions] = False

        current_theta = np.zeros(n_examinees, dtype=np.float64)
        for examinee in range(n_examinees):
            current_theta[examinee] = estimate_theta_single(
                raw_item_bank,
                raw_item_bank[selected_items[: step + 1, examinee]],
                responses[: step + 1, examinee],
                float(states[examinee, 0]),
            )

        theta_estimates[step] = current_theta
        states = np.column_stack(
            (
                current_theta.astype(np.float32),
                np.full(
                    n_examinees,
                    (step + 1) / config.test_length,
                    dtype=np.float32,
                ),
            )
        )

    errors = theta_estimates - true_theta[np.newaxis, :]
    summary = pd.DataFrame(
        {
            "step": np.arange(1, config.test_length + 1),
            "Bias": errors.mean(axis=1),
            "RMSE": np.sqrt(np.mean(errors**2, axis=1)),
            "MAE": np.mean(np.abs(errors), axis=1),
        }
    )
    return summary, selected_items, responses, theta_estimates


def make_test_records(
    selected_items: np.ndarray,
    responses: np.ndarray,
    theta_estimates: np.ndarray,
    true_theta: np.ndarray,
) -> pd.DataFrame:
    test_length, n_examinees = selected_items.shape
    errors = theta_estimates - true_theta[np.newaxis, :]
    return pd.DataFrame(
        {
            "userID": np.repeat(np.arange(1, n_examinees + 1), test_length),
            "step": np.tile(np.arange(1, test_length + 1), n_examinees),
            "itemID": (selected_items + 1).T.reshape(-1),
            "resp": responses.T.reshape(-1),
            "theta_est": theta_estimates.T.reshape(-1),
            "bias": errors.T.reshape(-1),
        }
    )

## Training

`state=[theta_hat_MLE, t/L]` は行動選択前の状態である。選択・回答後の
`next_state` は `[updated_theta_hat, (t+1)/L]` とする。



In [8]:
def train(
    config: Config,
    eval_net: DQNParamRaw,
    target_net: DQNParamRaw,
    raw_item_bank: np.ndarray,
    standardized_items: torch.Tensor,
) -> tuple[dict[str, torch.Tensor], pd.DataFrame]:
    if config.prior == "normal":
        training_theta = np.random.randn(config.training_size)
    elif config.prior == "uniform":
        training_theta = np.random.uniform(-3.0, 3.0, config.training_size)
    else:
        raise ValueError(f"Unsupported prior: {config.prior!r}.")

    replay_buffer = ReplayBuffer(
        config.memory_capacity, config.state_size, raw_item_bank.shape[0]
    )
    optimizer = optim.Adam(eval_net.parameters(), lr=config.learning_rate)
    loss_function = nn.MSELoss()
    best_validation_rmse = np.inf
    best_state: dict[str, torch.Tensor] | None = None
    learn_step_counter = 0
    interval_losses: list[float] = []
    train_log: list[dict[str, float | int]] = []

    eval_net.train()
    for episode, true_theta in enumerate(training_theta, start=1):
        # EXP029 MLE starts theta from Uniform(-0.5, 0.5); t/L starts at 0.
        state = np.array([np.random.rand() - 0.5, 0.0], dtype=np.float32)
        available = np.ones(raw_item_bank.shape[0], dtype=np.bool_)
        selected_items: list[int] = []
        observed_responses: list[int] = []

        for step in range(config.test_length):
            action = choose_action(
                eval_net,
                state,
                available,
                standardized_items,
                config.epsilon,
            )

            # Reward is evaluated at the pre-response MLE estimate.
            reward = float(
                fisher_information(raw_item_bank[action : action + 1], state[0])[0]
            )
            response = int(respond(raw_item_bank[action : action + 1], true_theta)[0])
            selected_items.append(action)
            observed_responses.append(response)

            theta_hat = estimate_theta_single(
                raw_item_bank,
                raw_item_bank[np.asarray(selected_items)],
                np.asarray(observed_responses),
                float(state[0]),
            )
            next_state = np.array(
                [theta_hat, (step + 1) / config.test_length], dtype=np.float32
            )
            next_available = available.copy()
            next_available[action] = False
            terminal = step + 1 == config.test_length

            replay_buffer.add(
                state,
                action,
                reward,
                next_state,
                terminal,
                next_available,
            )
            state = next_state
            available = next_available

            if len(replay_buffer) >= config.batch_size:
                loss = optimize_dqn(
                    config,
                    eval_net,
                    target_net,
                    optimizer,
                    loss_function,
                    replay_buffer,
                    standardized_items,
                )
                interval_losses.append(loss)
                learn_step_counter += 1
                if learn_step_counter % config.q_network_iteration == 0:
                    target_net.load_state_dict(eval_net.state_dict())

        if episode % config.validation_interval == 0:
            validation_theta = np.random.choice(
                training_theta, size=config.validation_size
            )
            validation_summary, _, _, _ = simulate_greedy_cat(
                config,
                eval_net,
                raw_item_bank,
                standardized_items,
                validation_theta,
            )
            validation_start_step = min(7, config.test_length)
            validation_rmse = float(
                validation_summary.loc[
                    validation_summary["step"] >= validation_start_step, "RMSE"
                ].mean()
            )
            mean_train_loss = (
                float(np.mean(interval_losses)) if interval_losses else np.nan
            )
            train_log.append(
                {
                    "episode": episode,
                    "train_loss": mean_train_loss,
                    "validation_rmse_step_7_40_mean": validation_rmse,
                }
            )
            interval_losses.clear()
            print(
                f"episode {episode:4d}: train_loss={mean_train_loss:.6f}, "
                f"validation_rmse={validation_rmse:.6f}"
            )

            if validation_rmse < best_validation_rmse:
                best_validation_rmse = validation_rmse
                best_state = copy.deepcopy(eval_net.state_dict())
            eval_net.train()

    if best_state is None:
        raise RuntimeError("No checkpoint was selected during validation.")
    return best_state, pd.DataFrame(train_log)


eval_net = DQNParamRaw(
    cfg.state_size,
    cfg.item_feature_size,
    cfg.first_hidden,
    cfg.second_hidden,
).to(DEVICE)
target_net = DQNParamRaw(
    cfg.state_size,
    cfg.item_feature_size,
    cfg.first_hidden,
    cfg.second_hidden,
).to(DEVICE)
eval_net.initialize()
target_net.load_state_dict(eval_net.state_dict())

best_state, train_log = train(
    cfg, eval_net, target_net, item_bank, item_features_tensor
)
eval_net.load_state_dict(best_state)

episode   50: train_loss=0.018058, validation_rmse=0.450906
episode  100: train_loss=0.000686, validation_rmse=0.476692
episode  150: train_loss=0.000382, validation_rmse=0.415635
episode  200: train_loss=0.000291, validation_rmse=0.437205
episode  250: train_loss=0.000289, validation_rmse=0.428776
episode  300: train_loss=0.000218, validation_rmse=0.424706
episode  350: train_loss=0.000165, validation_rmse=0.414868
episode  400: train_loss=0.000158, validation_rmse=0.428989
episode  450: train_loss=0.000131, validation_rmse=0.444469
episode  500: train_loss=0.000136, validation_rmse=0.397543
episode  550: train_loss=0.000090, validation_rmse=0.425688
episode  600: train_loss=0.000098, validation_rmse=0.440194
episode  650: train_loss=0.000094, validation_rmse=0.434316
episode  700: train_loss=0.000093, validation_rmse=0.397361
episode  750: train_loss=0.000079, validation_rmse=0.423513
episode  800: train_loss=0.000076, validation_rmse=0.396443
episode  850: train_loss=0.000063, valid

<All keys matched successfully>

## Save model and test results

チェックポイントにはモデル重みだけでなくConfigと項目標準化係数を保存する。



In [9]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

stem = (
    f"uncor_{cfg.bank_id}_DQN_Param_Raw_MLE_{cfg.prior}"
    f"_gamma_{cfg.gamma}_seed_{cfg.seed}"
)
model_path = (
    MODEL_DIR / f"dqn_param_raw_MLE_{cfg.prior}_uncor_{cfg.bank_id}"
    f"_gamma_{cfg.gamma}_seed_{cfg.seed}.pt"
)
torch.save(
    {
        "model_state_dict": eval_net.state_dict(),
        "config": asdict(cfg),
        "item_feature_mean": item_feature_mean,
        "item_feature_std": item_feature_std,
    },
    model_path,
)
train_log.to_csv(RESULTS_DIR / f"train_log_{stem}.csv", index=False)
print(f"Model saved to: {model_path}")

test_summary, test_items, test_responses, test_theta_estimates = simulate_greedy_cat(
    cfg,
    eval_net,
    item_bank,
    item_features_tensor,
    theta_test,
    response_seed=cfg.seed,
)
test_records = make_test_records(
    test_items, test_responses, test_theta_estimates, theta_test
)
test_records.to_csv(RESULTS_DIR / f"records_{stem}.csv", index=False)
test_summary.to_csv(RESULTS_DIR / f"summary_{stem}.csv", index=False)

print(test_summary.to_string(index=False))
print(f"Results saved to: {RESULTS_DIR}")

Model saved to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP029/models/dqn_param_raw_MLE_normal_uncor_1_gamma_0.1_seed_42.pt
 step     Bias     RMSE      MAE
    1 0.326423 1.308386 1.082785
    2 0.277486 1.168178 0.932566
    3 0.190174 1.032165 0.807934
    4 0.008978 1.171957 0.852654
    5 0.119925 0.876525 0.674350
    6 0.097341 0.848579 0.640982
    7 0.090034 0.801052 0.598573
    8 0.091342 0.727477 0.553643
    9 0.073694 0.713883 0.527732
   10 0.072806 0.649904 0.491291
   11 0.062694 0.625133 0.470277
   12 0.057845 0.591137 0.447960
   13 0.051963 0.562938 0.427779
   14 0.045200 0.543119 0.410550
   15 0.039664 0.520266 0.396034
   16 0.031537 0.507575 0.382696
   17 0.028676 0.482283 0.367903
   18 0.029146 0.466553 0.357196
   19 0.028540 0.452043 0.346507
   20 0.025741 0.439169 0.335571
   21 0.024183 0.428189 0.326148
   22 0.022185 0.414773 0.317874
   23 0.020543 0.404009 0.312055
   24 0.021453 0.396850 0.306034